# Blelloch Prefix Scan for Mamba2 — GPU Test Suite

**Issue**: state-spaces/mamba#716 — Mamba2 chunked scan has systematic PPL bias

**Solution**: Replace chunked scan with mathematically exact Blelloch (tree) prefix scan

## Instructions
1. Run all cells in order
2. Estimated time: ~10 min
3. Requires GPU (T4/P100) — select Runtime → Change runtime type → GPU

In [ ]:
import sys, subprocess, os

# Clone branch with Blelloch implementation
REPO = "https://github.com/KakashiTech/mamba"
BRANCH = "main"
DEST = "/kaggle/working/mamba"

if not os.path.exists(DEST):
    !git clone -b {BRANCH} {REPO} {DEST}
%cd {DEST}
!pip install -e . -q
!pip install einops pytest -q

sys.path.insert(0, os.path.join(DEST, "mamba_ssm/ops"))

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
%%bash
cd /kaggle/working/mamba && python -m pytest tests/test_blelloch_integration.py -v 2>&1

---
## PPL Benchmark (requires Mamba2 model)

The next cell monkey-patches Mamba2's selective_scan_fn and measures perplexity.
Skip if running out of VRAM (T4: ~15GB usable, enough for mamba2-370M in fp16).

In [ ]:
!python bench_blelloch_ppl.py --model mamba2-370m --dtype float16 2>&1 | tail -30

In [ ]:
print("All tests passed. PPL benchmark complete.")